In [1]:
"""
Foreground-Biased PCA Fusion (FB-PCA)
======================================
Standard PCA is dominated by background pixels (poles occupy <5% of image area).
FB-PCA fits PCA *exclusively* on foreground pixels (inside bounding boxes),
so the principal components capture variance that discriminates poles across
all 4 modalities — including range.

Key difference from Early-PCA 3:
  - Early-PCA 3: PCA fitted on ALL pixels → background-dominated components
  - FB-PCA      : PCA fitted on FOREGROUND pixels only → pole-discriminative components

The resulting 3-ch images are then used for standard YOLOv11n training.

Output: dataset_fbpca/images/{train,valid,test}/*.png
"""

import cv2
import numpy as np
from pathlib import Path
from sklearn.decomposition import PCA
from tqdm import tqdm

# ── CONFIG ───────────────────────────────────────────────────
ROOT = Path(
    "SnowPole Detection A Comprehensive Dataset for Detection and Localization "
    "Using LiDAR Imaging in Nordic Winter Conditions"
) / "SnowPole_Detection_Dataset"

LABEL_ROOT  = ROOT / "labels"
OUT_ROOT    = Path("dataset_fbpca/images")
MODALITIES  = ["reflec", "signal", "nearir", "range"]
SPLITS      = ["train", "valid", "test"]
IMG_EXTS    = {".png", ".jpg", ".jpeg"}

# Sampling config
FG_SAMPLES_PER_IMAGE = 800   # foreground pixels per image (poles are rare — oversample)
MAX_PIXELS           = 300_000
BBOX_DILATE          = 3     # slight dilation to capture pole edges
np.random.seed(42)
# ─────────────────────────────────────────────────────────────


def read_channel(path: Path) -> np.ndarray:
    img = cv2.imread(str(path), cv2.IMREAD_UNCHANGED)
    if img is None:
        raise FileNotFoundError(path)
    if img.ndim == 3:
        img = img[:, :, 0]
    return img.astype(np.float32)


def load_stack(img_name: str, split: str) -> np.ndarray:
    """Returns H x W x 4."""
    return np.stack(
        [read_channel(ROOT / mod / split / img_name) for mod in MODALITIES],
        axis=-1
    )


def parse_labels(label_path: Path, H: int, W: int):
    """Parse YOLO format labels → list of (x1,y1,x2,y2) pixel boxes."""
    boxes = []
    if not label_path.exists():
        return boxes
    with open(label_path) as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) < 5:
                continue
            _, cx, cy, bw, bh = map(float, parts[:5])
            x1 = max(0, int((cx - bw / 2) * W) - BBOX_DILATE)
            y1 = max(0, int((cy - bh / 2) * H) - BBOX_DILATE)
            x2 = min(W, int((cx + bw / 2) * W) + BBOX_DILATE)
            y2 = min(H, int((cy + bh / 2) * H) + BBOX_DILATE)
            if x2 > x1 and y2 > y1:
                boxes.append((x1, y1, x2, y2))
    return boxes


def fg_mask(boxes, H: int, W: int) -> np.ndarray:
    """Build binary foreground mask from bounding boxes."""
    mask = np.zeros((H, W), dtype=bool)
    for x1, y1, x2, y2 in boxes:
        mask[y1:y2, x1:x2] = True
    return mask


def per_channel_normalize(stack: np.ndarray) -> np.ndarray:
    """Normalize each channel to [0,1] using per-image min/max."""
    normed = np.zeros_like(stack)
    for c in range(stack.shape[-1]):
        ch = stack[:, :, c]
        mn, mx = ch.min(), ch.max()
        normed[:, :, c] = (ch - mn) / (mx - mn + 1e-6)
    return normed


# ══════════════════════════════════════════════════════════════
# STEP 1 — Collect FOREGROUND pixels from training set
# ══════════════════════════════════════════════════════════════
print("=" * 60)
print("Foreground-Biased PCA Fusion (FB-PCA)")
print("=" * 60)
print("\nSTEP 1: Collecting foreground pixels from training set...")

train_ref_dir = ROOT / MODALITIES[0] / "train"
train_imgs    = [p for p in train_ref_dir.iterdir() if p.suffix.lower() in IMG_EXTS]

fg_pixels  = []
n_skipped  = 0

for img_path in tqdm(train_imgs, desc="Sampling FG"):
    stack  = load_stack(img_path.name, "train")
    normed = per_channel_normalize(stack)
    H, W   = stack.shape[:2]

    label_path = LABEL_ROOT / "train" / (img_path.stem + ".txt")
    boxes = parse_labels(label_path, H, W)

    if not boxes:
        n_skipped += 1
        continue

    mask   = fg_mask(boxes, H, W)
    pixels = normed[mask]        # N_fg x 4

    if len(pixels) == 0:
        continue

    # Oversample foreground — poles are small, we need enough samples
    n = min(len(pixels), FG_SAMPLES_PER_IMAGE)
    idx = np.random.choice(len(pixels), n, replace=len(pixels) < n)
    fg_pixels.append(pixels[idx])

fg_pixels = np.concatenate(fg_pixels)
print(f"  Images with labels : {len(train_imgs) - n_skipped}/{len(train_imgs)}")
print(f"  Total FG pixels    : {len(fg_pixels)}")

if len(fg_pixels) > MAX_PIXELS:
    idx = np.random.choice(len(fg_pixels), MAX_PIXELS, replace=False)
    fg_pixels = fg_pixels[idx]
    print(f"  Subsampled to      : {len(fg_pixels)}")


# ══════════════════════════════════════════════════════════════
# STEP 2 — Fit PCA on foreground pixels only
# ══════════════════════════════════════════════════════════════
print("\nSTEP 2: Fitting PCA on foreground pixels (4 → 3)...")

pca = PCA(n_components=3)
pca.fit(fg_pixels)

print(f"  Explained variance : {np.round(pca.explained_variance_ratio_, 4)}")
print(f"  Total captured     : {pca.explained_variance_ratio_.sum():.4f}")
print(f"\n  PCA components (how each modality contributes):")
modality_names = MODALITIES
for i, comp in enumerate(pca.components_):
    contributions = {m: f"{v:+.3f}" for m, v in zip(modality_names, comp)}
    print(f"    PC{i+1}: {contributions}")


# ══════════════════════════════════════════════════════════════
# STEP 3 — Global normalisation stats (from train projections)
# ══════════════════════════════════════════════════════════════
print("\nSTEP 3: Computing global normalisation stats from train...")

global_lo = np.full(3,  np.inf)
global_hi = np.full(3, -np.inf)

for img_path in tqdm(train_imgs, desc="Norm stats"):
    stack  = load_stack(img_path.name, "train")
    normed = per_channel_normalize(stack)
    proj   = pca.transform(normed.reshape(-1, 4)).reshape(
        normed.shape[0], normed.shape[1], 3
    )
    for c in range(3):
        global_lo[c] = min(global_lo[c], np.percentile(proj[:,:,c],  1))
        global_hi[c] = max(global_hi[c], np.percentile(proj[:,:,c], 99))

print(f"  Global lo: {np.round(global_lo, 4)}")
print(f"  Global hi: {np.round(global_hi, 4)}")

# Save transform params
np.save("fbpca_components.npy", pca.components_)
np.save("fbpca_mean.npy",       pca.mean_)
np.save("fbpca_global_lo.npy",  global_lo)
np.save("fbpca_global_hi.npy",  global_hi)
print("  Saved: fbpca_components.npy, fbpca_mean.npy, fbpca_global_lo.npy, fbpca_global_hi.npy")


# ══════════════════════════════════════════════════════════════
# STEP 4 — Write fused images for all splits
# ══════════════════════════════════════════════════════════════
def fuse(img_name: str, split: str) -> np.ndarray:
    """4-ch stack → 3-ch BGR uint8 via FB-PCA."""
    stack  = load_stack(img_name, split)
    normed = per_channel_normalize(stack)
    proj   = pca.transform(normed.reshape(-1, 4)).reshape(
        normed.shape[0], normed.shape[1], 3
    )
    def norm_ch(ch, lo, hi):
        return ((np.clip(ch, lo, hi) - lo) / (hi - lo + 1e-6) * 255).astype(np.uint8)

    r = norm_ch(proj[:,:,0], global_lo[0], global_hi[0])
    g = norm_ch(proj[:,:,1], global_lo[1], global_hi[1])
    b = norm_ch(proj[:,:,2], global_lo[2], global_hi[2])
    return cv2.merge([b, g, r])   # OpenCV BGR


print("\nSTEP 4: Writing fused images...")

for split in SPLITS:
    out_dir = OUT_ROOT / split
    out_dir.mkdir(parents=True, exist_ok=True)

    ref_dir    = ROOT / MODALITIES[0] / split
    split_imgs = [p for p in ref_dir.iterdir() if p.suffix.lower() in IMG_EXTS]

    for img_path in tqdm(split_imgs, desc=split):
        fused = fuse(img_path.name, split)
        cv2.imwrite(str(out_dir / (img_path.stem + ".png")), fused)

print(f"\n✅ Done. Dataset saved to: {OUT_ROOT}")
print("""
Summary — what makes FB-PCA different:
  • PCA fitted on FOREGROUND pixels only (inside bounding boxes)
  • Components capture inter-modal variance AT pole locations
  • Range contribution in PC components reflects how range
    distinguishes poles from background — not just image-wide variance
  • Global normalisation ensures consistent color space across splits
""")

Foreground-Biased PCA Fusion (FB-PCA)

STEP 1: Collecting foreground pixels from training set...


Sampling FG: 100%|██████████| 1367/1367 [00:29<00:00, 45.65it/s]


  Images with labels : 1367/1367
  Total FG pixels    : 977106
  Subsampled to      : 300000

STEP 2: Fitting PCA on foreground pixels (4 → 3)...
  Explained variance : [0.596  0.2623 0.1232]
  Total captured     : 0.9815

  PCA components (how each modality contributes):
    PC1: {'reflec': '+0.706', 'signal': '+0.683', 'nearir': '+0.173', 'range': '+0.064'}
    PC2: {'reflec': '-0.334', 'signal': '+0.117', 'nearir': '+0.932', 'range': '-0.079'}
    PC3: {'reflec': '-0.607', 'signal': '+0.719', 'nearir': '-0.318', 'range': '-0.116'}

STEP 3: Computing global normalisation stats from train...


Norm stats: 100%|██████████| 1367/1367 [00:31<00:00, 43.17it/s]


  Global lo: [-0.6168 -0.8246 -0.6241]
  Global hi: [0.9169 0.3856 0.4903]
  Saved: fbpca_components.npy, fbpca_mean.npy, fbpca_global_lo.npy, fbpca_global_hi.npy

STEP 4: Writing fused images...


test: 100%|██████████| 197/197 [00:05<00:00, 34.70it/s]


✅ Done. Dataset saved to: dataset_fbpca\images

Summary — what makes FB-PCA different:
  • PCA fitted on FOREGROUND pixels only (inside bounding boxes)
  • Components capture inter-modal variance AT pole locations
  • Range contribution in PC components reflects how range
    distinguishes poles from background — not just image-wide variance
  • Global normalisation ensures consistent color space across splits



In [2]:
from ultralytics import YOLO

print("Starting YOLO training with 4-channel input...")

model = YOLO("yolo11n.yaml")

model.train(
    data="correlation_maps.yaml",
    imgsz=1024,
    epochs=500,
    patience=80,        # early stopping
    batch=8,
    device=0,
    project="correlation_maps",
    name="correlation_maps",
    amp=False,
    augment=False,
    workers=0,
)

Starting YOLO training with 4-channel input...
New https://pypi.org/project/ultralytics/8.4.21 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.6  Python-3.11.0 torch-2.9.1+cu130 CUDA:0 (NVIDIA GeForce RTX 5050 Laptop GPU, 8151MiB)
engine\trainer: agnostic_nms=False, amp=False, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=correlation_maps.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=500, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=1024, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.yaml, momentum=0.937, m

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x0000021F155FB210>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.0480